# DNA Token Analysis for LLM Tokenizers

Check if Mixtral/Qwen tokenizers have DNA-like tokens (sequences of only A, T, C, G uppercase) and compare to NatueLM

In [1]:
from transformers import AutoTokenizer
import re

/fsx/loubna/projects_v2/tokenize-data/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def find_dna_tokens(tokenizer):
    """Find all tokens consisting only of uppercase A, T, C, G"""
    dna_pattern = re.compile(r'^[ATCG]+$')
    dna_tokens = []
    
    vocab = tokenizer.get_vocab()
    for token, idx in vocab.items():
        # Clean token (some tokenizers add special chars like Ġ for space)
        clean = token.replace('Ġ', '').replace('▁', '')
        if dna_pattern.match(clean) and len(clean) > 0:
            dna_tokens.append((token, clean, idx, len(clean)))
    
    # Sort by length
    dna_tokens.sort(key=lambda x: (-x[3], x[1]))
    return dna_tokens

## 1. Mixtral Tokenizer

In [3]:
mixtral_tok = AutoTokenizer.from_pretrained("mistralai/Mixtral-8x7B-v0.1")
print(f"Vocab size: {len(mixtral_tok.get_vocab())}")

Vocab size: 32000


In [4]:
mixtral_dna = find_dna_tokens(mixtral_tok)
print(f"Found {len(mixtral_dna)} DNA-like tokens in Mixtral:\n")
for token, clean, idx, length in mixtral_dna:
    print(f"  {repr(token):20} -> {clean:10} (id={idx}, len={length})")

Found 41 DNA-like tokens in Mixtral:

  'AAAAAAAA'           -> AAAAAAAA   (id=21844, len=8)
  'AAAA'               -> AAAA       (id=11086, len=4)
  'ACC'                -> ACC        (id=26749, len=3)
  '▁ACT'               -> ACT        (id=26535, len=3)
  'ACT'                -> ACT        (id=7637, len=3)
  'ATA'                -> ATA        (id=5854, len=3)
  'ATT'                -> ATT        (id=18501, len=3)
  'TAG'                -> TAG        (id=12137, len=3)
  'AA'                 -> AA         (id=3598, len=2)
  '▁AA'                -> AA         (id=24724, len=2)
  'AC'                 -> AC         (id=1645, len=2)
  '▁AC'                -> AC         (id=11949, len=2)
  'AG'                 -> AG         (id=2377, len=2)
  '▁AG'                -> AG         (id=14778, len=2)
  'AT'                 -> AT         (id=962, len=2)
  '▁AT'                -> AT         (id=9274, len=2)
  'CA'                 -> CA         (id=5194, len=2)
  '▁CA'                -> CA        

In [5]:
# Test how Mixtral tokenizes a DNA sequence
test_dna = "GGTGATTCATTGTCATTTTCTATTGTTTCATATTCATCCGACATATCTAGACCAGTTGCAGGGTCCTTTCGT"
tokens = mixtral_tok.tokenize(test_dna)
print(f"Mixtral tokenizes '{test_dna}' as:")
print(tokens)

Mixtral tokenizes 'GGTGATTCATTGTCATTTTCTATTGTTTCATATTCATCCGACATATCTAGACCAGTTGCAGGGTCCTTTCGT' as:
['▁G', 'GT', 'G', 'AT', 'TC', 'ATT', 'G', 'TC', 'AT', 'TT', 'T', 'CT', 'ATT', 'GT', 'T', 'TC', 'AT', 'AT', 'TC', 'AT', 'CC', 'G', 'AC', 'AT', 'AT', 'CT', 'AG', 'ACC', 'AG', 'TT', 'GC', 'AG', 'GG', 'TC', 'CT', 'T', 'TC', 'GT']


## 2. Qwen3 Tokenizer

In [6]:
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Base")
print(f"Vocab size: {len(qwen_tok.get_vocab())}")

Vocab size: 151669


In [7]:
qwen_dna = find_dna_tokens(qwen_tok)
print(f"Found {len(qwen_dna)} DNA-like tokens in Qwen3:\n")
for token, clean, idx, length in qwen_dna:
    print(f"  {repr(token):20} -> {clean:10} (id={idx}, len={length})")

Found 72 DNA-like tokens in Qwen3:

  'AAAAAAAA'           -> AAAAAAAA   (id=57905, len=8)
  'CCCCCC'             -> CCCCCC     (id=91443, len=6)
  'AAAA'               -> AAAA       (id=25699, len=4)
  'CCCC'               -> CCCC       (id=47932, len=4)
  'AAA'                -> AAA        (id=50107, len=3)
  'ĠAAA'               -> AAA        (id=47097, len=3)
  'ĠAAC'               -> AAC        (id=75348, len=3)
  'AAC'                -> AAC        (id=96710, len=3)
  'ĠACA'               -> ACA        (id=64938, len=3)
  'ACA'                -> ACA        (id=62538, len=3)
  'ĠACC'               -> ACC        (id=25840, len=3)
  'ACC'                -> ACC        (id=29442, len=3)
  'ĠACT'               -> ACT        (id=21116, len=3)
  'ACT'                -> ACT        (id=6823, len=3)
  'ATA'                -> ATA        (id=4485, len=3)
  'ĠATA'               -> ATA        (id=91622, len=3)
  'ATT'                -> ATT        (id=21614, len=3)
  'ĠATT'               -> ATT  

In [8]:
# Test how Qwen tokenizes a DNA sequence  
tokens = qwen_tok.tokenize(test_dna)
print(f"Qwen3 tokenizes '{test_dna}' as:")
print(tokens)

Qwen3 tokenizes 'GGTGATTCATTGTCATTTTCTATTGTTTCATATTCATCCGACATATCTAGACCAGTTGCAGGGTCCTTTCGT' as:
['GG', 'TG', 'AT', 'TC', 'ATT', 'G', 'TC', 'AT', 'TT', 'T', 'CT', 'ATT', 'G', 'TT', 'TC', 'AT', 'AT', 'TC', 'AT', 'CC', 'G', 'AC', 'AT', 'AT', 'CT', 'AG', 'ACC', 'AG', 'TT', 'GC', 'AG', 'GG', 'TC', 'CT', 'T', 'TC', 'GT']


## 3. NatureLM Tokenizer

How it works:
- Special wrapper tags: DNA sequences must be wrapped in <dna>...</dna> (character-level) or <dna6mer>...</dna6mer> (6-mer chunking) tags - the tokenizer uses these to know when to switch from BPE to DNA tokenization mode (6-mer with character level fallback)

- <d> prefix on all DNA tokens: Every DNA token gets a <d> prefix - both single nucleotides (<d>A, <d>T, <d>C, <d>G) and 6-mers (<d>ATCGAT). This gives DNA a separate embedding space from English letters in case there's an overlap

- 6-mer chunking with fallback: In dna6mer mode, it does 6-mer chunking, falling back to single characters when no 6-mer match exists in case of ambiguous nucleotides or not lengths divisible by 6

In [ ]:
# NatureLM has custom tokenizer class
naturelm_tok = AutoTokenizer.from_pretrained("microsoft/NatureLM-8x7B", trust_remote_code=True)
print(f"Vocab size: {len(naturelm_tok.get_vocab())}")

Vocab size: 38078


In [10]:
# Test DNA tokenization with <dna> tags
test_with_tag = "<dna>ATCGATCGATCGATCGATGCC</dna>"
tokens = naturelm_tok.tokenize(test_with_tag)
print(f"NatureLM tokenizes '{test_with_tag}' as:")
print(tokens)

# Test with dna6mer tag
test_6mer = "<dna6mer>ATCGATCGATCGATCG</dna6mer>"
tokens_6mer = naturelm_tok.tokenize(test_6mer)
print(f"\nNatureLM tokenizes '{test_6mer}' as:")
print(tokens_6mer)

NatureLM tokenizes '<dna>ATCGATCGATCGATCGATGCC</dna>' as:
['<dna>', '<d>A', '<d>T', '<d>C', '<d>G', '<d>A', '<d>T', '<d>C', '<d>G', '<d>A', '<d>T', '<d>C', '<d>G', '<d>A', '<d>T', '<d>C', '<d>G', '<d>A', '<d>T', '<d>G', '<d>C', '<d>C', '</dna>']

NatureLM tokenizes '<dna6mer>ATCGATCGATCGATCG</dna6mer>' as:
['<dna6mer>', '<d>ATCGAT', '<d>CGATCG', '<d>A', '<d>T', '<d>C', '<d>G', '</dna6mer>']


In [12]:
# Check what DNA tokens exist in NatureLM
naturelm_dna = find_dna_tokens(naturelm_tok)
print(f"Found {len(naturelm_dna)} pure DNA-like tokens (no <d> prefix)")

# Also check for <d> prefixed tokens
vocab = naturelm_tok.get_vocab()
d_prefixed = [(t, idx) for t, idx in vocab.items() if t.startswith('<d>')]
print(f"Found {len(d_prefixed)} <d>-prefixed tokens")
print(f"Examples: {d_prefixed[:20]}")

Found 41 pure DNA-like tokens (no <d> prefix)
Found 4112 <d>-prefixed tokens
Examples: [('<d>A', 33899), ('<d>C', 33901), ('<d>T', 33903), ('<d>G', 33905), ('<d>U', 33907), ('<d>R', 33909), ('<d>Y', 33911), ('<d>S', 33913), ('<d>W', 33915), ('<d>K', 33917), ('<d>M', 33919), ('<d>B', 33921), ('<d>D', 33923), ('<d>H', 33925), ('<d>V', 33927), ('<d>N', 33929), ('<d>AAAAAA', 33982), ('<d>AAAAAT', 33983), ('<d>AAAAAC', 33984), ('<d>AAAAAG', 33985)]


## 4. Compare tokenization efficiency

In [13]:
# Compare how many tokens each model needs for same DNA sequence
long_dna = "ATCGATCGATCG" * 10  # 120 nucleotides

print(f"DNA sequence length: {len(long_dna)} nucleotides\n")

mixtral_tokens = mixtral_tok.tokenize(long_dna)
print(f"Mixtral: {len(mixtral_tokens)} tokens")

qwen_tokens = qwen_tok.tokenize(long_dna)
print(f"Qwen3:   {len(qwen_tokens)} tokens")

if naturelm_tok:
    naturelm_tokens = naturelm_tok.tokenize(f"<dna6mer>{long_dna}</dna6mer>")
    # Subtract the tag tokens
    print(f"NatureLM (6mer): {len(naturelm_tokens)} tokens (includes tags)")

DNA sequence length: 120 nucleotides

Mixtral: 61 tokens
Qwen3:   60 tokens
NatureLM (6mer): 22 tokens (includes tags)
